# ======================================================================
# MODULE 6: ChaCha20-Poly1305 Decryption & Verification (Receiver Side)
# ======================================================================
#
# **Project:** QVSC — Quantum-assisted Video Steganographic Communication
#
# **Author:** Hasibul Hasan (Roll: 2003127, CSE, RUET)
#
# **Purpose:** Decrypt the ciphertext recovered by Module 5 using the
# quantum-derived key from Module 1. This is the final cryptographic
# step at the receiver side — it both decrypts the message AND verifies
# its integrity via the Poly1305 authentication tag.
#
# **This is the integrity-verification gate of the QVSC pipeline.**
# Lossy H.264 compression leaves a small residual bit-error rate, so the
# extracted payload is first passed through the Reed-Solomon FEC layer
# (fec_module.fec_decode), which corrects those errors and recovers the
# bit-exact ciphertext. Any error that FEC CANNOT correct will then cause
# the Poly1305 tag check to fail. ChaCha20-Poly1305 is an AEAD cipher
# (Authenticated Encryption with Associated Data), which means it
# rejects tampered ciphertext outright — no partial decryption, no
# "looks mostly right" output. Either everything is recovered exactly,
# or the receiver is told the data is corrupted.
#
# **Why this guarantees end-to-end security:**
#   1. The 256-bit key was generated by E91 QKD — any eavesdropping
#      attempt would have been caught by the CHSH test in Module 1.
#   2. The Poly1305 tag is computed over (Ciphertext + AAD), so any
#      modification — to the embedded bits, to the cover video, to
#      the metadata — breaks verification.
#   3. The AAD `b"QVSC-Module2"` binds the ciphertext to this protocol,
#      preventing cross-protocol replay attacks.
#
# **Input:**
# - `extracted_ciphertext.bin` — From Module 5 (FEC-protected payload:
#                                 RS codewords, interleaved)
# - `quantum_key.bin`           — From Module 1 (32-byte symmetric key)
#
# **Output:**
# - `recovered_message.txt` (if text) OR `recovered_image.<ext>` (if image)
# - `decryption_report.npy` — Diagnostic information for Module 7
#
# **Algorithm:**
# ```
# 1. Load 256-bit key from quantum_key.bin
# 2. Load extracted_ciphertext.bin → Reed-Solomon FEC-decode (de-interleave
#    + RS-correct) to recover the bit-exact ciphertext
# 2b. Parse the recovered ciphertext → Nonce, Ciphertext, Tag
# 3. Initialize ChaCha20-Poly1305 with (key, nonce)
# 4. Update with AAD = b"QVSC-Module2"
# 5. decrypt_and_verify(ciphertext, tag):
#    a. Recompute Poly1305 tag over (AAD || Ciphertext)
#    b. Constant-time compare with received tag
#    c. If match    → produce plaintext
#    d. If mismatch → raise ValueError (REJECT)
# 6. Parse 4-byte type header:
#    0x00000001 → text → decode UTF-8 → save .txt
#    0x00000002 → image → save raw bytes to file
# ```
#
# **Symmetry with Module 2:** This module is the exact inverse of
# Module 2. Same key, same AAD, same nonce, same tag → original
# plaintext. If any of those four inputs differs, decryption fails.
#
# **Tools:** PyCryptodome (`Crypto.Cipher.ChaCha20_Poly1305`)
#
# ======================================================================

## Section 1: Library Imports

In [2]:
from Crypto.Cipher import ChaCha20_Poly1305
import struct
import os
import numpy as np
import hashlib

print("All libraries imported successfully.")

All libraries imported successfully.


## Section 2: Configuration

The receiver only needs three things:
1. The recovered ciphertext blob (from Module 5)
2. The quantum-derived key (from Module 1)
3. The AAD that was used during encryption (must match Module 2)

The AAD acts as a protocol binding: even if an attacker had the right
key and ciphertext, they could not produce a valid tag without knowing
the exact AAD string. We use `b"QVSC-Module2"` to match Module 2.

In [3]:
# ── File paths (inputs) ───────────────────────────────────────────────
EXTRACTED_CIPHERTEXT = 'extracted_ciphertext.bin'   # From Module 5
KEY_FILE             = 'quantum_key.bin'            # From Module 1

# ── Output paths ──────────────────────────────────────────────────────
OUTPUT_TEXT_FILE     = 'recovered_message.txt'      # If payload is text
OUTPUT_IMAGE_FILE    = 'recovered_image.bin'        # If payload is image
DECRYPTION_REPORT    = 'decryption_report.npy'      # Diagnostic output

# ── Cryptographic parameters (MUST match Module 2 exactly) ───────────
AAD = b"QVSC-Module2"     # Associated Authenticated Data — protocol tag

# ── Type headers (must match Module 2) ───────────────────────────────
TYPE_TEXT  = 0x00000001   # 4-byte big-endian header for text payload
TYPE_IMAGE = 0x00000002   # 4-byte big-endian header for image payload

# ── Optional: original plaintext for end-to-end validation ───────────
# In a real deployment the receiver does NOT have this. We use it only
# for thesis-experiment verification (was the round-trip lossless?).
ORIGINAL_PLAINTEXT_FILE = None    # e.g. 'original_secret.txt' or None

print("="*70)
print("  MODULE 6 CONFIGURATION (Receiver Side)")
print("="*70)
print(f"  Extracted ciphertext:  {EXTRACTED_CIPHERTEXT}")
print(f"  Quantum key:           {KEY_FILE}")
print(f"  AAD:                   {AAD!r}")
print(f"  Output (text):         {OUTPUT_TEXT_FILE}")
print(f"  Output (image):        {OUTPUT_IMAGE_FILE}")
print("="*70)

  MODULE 6 CONFIGURATION (Receiver Side)
  Extracted ciphertext:  extracted_ciphertext.bin
  Quantum key:           quantum_key.bin
  AAD:                   b'QVSC-Module2'
  Output (text):         recovered_message.txt
  Output (image):        recovered_image.bin


## Section 3: Load the Quantum-Derived Key

The 256-bit key was generated by Module 1 from E91 entangled qubit
measurements, sifted using compatible bases, and hashed through SHA3-256
to produce a uniform 32-byte symmetric key.

For ChaCha20-Poly1305, the key must be exactly 32 bytes (256 bits).
The Poly1305 sub-key is derived internally from the ChaCha20 keystream,
so we don't need to handle it separately.

In [4]:
# ── Load the quantum-derived key ──────────────────────────────────────
if not os.path.exists(KEY_FILE):
    raise FileNotFoundError(
        f"Key file '{KEY_FILE}' not found.\n"
        f"Module 1 must be run first to generate the quantum key."
    )

with open(KEY_FILE, 'rb') as f:
    key = f.read()

assert len(key) == 32, (
    f"Key must be exactly 32 bytes (256 bits) for ChaCha20-Poly1305, "
    f"got {len(key)} bytes."
)

# Compute SHA-256 fingerprint for diagnostic identification (not used in crypto)
key_fingerprint = hashlib.sha256(key).hexdigest()[:16]

print(f"Quantum key loaded from '{KEY_FILE}'")
print(f"  Key length:      {len(key)} bytes ({len(key) * 8} bits)")
print(f"  Key (hex):       {key.hex()}")
print(f"  Key fingerprint: {key_fingerprint} (SHA-256 prefix)")

Quantum key loaded from 'quantum_key.bin'
  Key length:      32 bytes (256 bits)
  Key (hex):       601a007182022fe6fa1a751282c76f67350e1cf85ec119bcab03240ad613d791
  Key fingerprint: 0deecb03eeeb9ce0 (SHA-256 prefix)


## Section 4: Load and Parse the Extracted Ciphertext

Module 5 wrote the recovered ciphertext to disk in the exact same
binary format as Module 2's output:

```
┌────────────────┬──────────────────────────┬─────────────────┐
│ Nonce (12 B)   │ Ciphertext (variable)    │ Poly1305 tag    │
│                │                          │ (16 B)          │
└────────────────┴──────────────────────────┴─────────────────┘
   Bytes 0..11      Bytes 12..N-17            Bytes N-16..N-1
```

The nonce was generated randomly by Module 2 during encryption — it
travels with the ciphertext (in the clear) because nonce confidentiality
is not required, only nonce uniqueness per (key, nonce) pair.

The 16-byte tag is the Poly1305 authentication code that protects
both the ciphertext and the AAD.

In [5]:
# ── Load the extracted ciphertext blob from Module 5 ─────────────────
if not os.path.exists(EXTRACTED_CIPHERTEXT):
    raise FileNotFoundError(
        f"Extracted ciphertext '{EXTRACTED_CIPHERTEXT}' not found.\n"
        f"Module 5 must be run first to recover the ciphertext."
    )

import fec_module as fec
with open(EXTRACTED_CIPHERTEXT, 'rb') as f:
    raw = f.read()

# Reed-Solomon decode FIRST: Module 5 hands us the FEC-protected payload
# (RS + interleaving), not the raw ciphertext. This corrects the residual
# channel errors so ChaCha20-Poly1305 receives a bit-exact ciphertext.
blob, fec_failures = fec.fec_decode(raw)
print(f"  FEC decode: {fec_failures} uncorrectable codeword(s) "
      f"-> {'bit-exact ciphertext recovered' if fec_failures == 0 else 'message may fail tag check'}")

# Sanity check: blob must be at least Nonce(12) + Tag(16) = 28 bytes
if len(blob) < 28:
    raise ValueError(
        f"Extracted ciphertext is only {len(blob)} bytes — minimum is 28 "
        f"(12-byte nonce + 16-byte tag). The extraction probably failed."
    )

# ── Parse the binary structure: Nonce(12) || Ciphertext(N) || Tag(16) ─
recv_nonce      = blob[:12]
recv_ciphertext = blob[12:-16]
recv_tag        = blob[-16:]

print("="*70)
print("  PARSED CIPHERTEXT BLOB")
print("="*70)
print(f"  File:               {EXTRACTED_CIPHERTEXT}")
print(f"  Total size:         {len(blob)} bytes")
print(f"  Nonce (hex):        {recv_nonce.hex()}")
print(f"  Nonce length:       {len(recv_nonce)} bytes")
print(f"  Ciphertext length:  {len(recv_ciphertext)} bytes")
print(f"  Ciphertext head:    {recv_ciphertext[:16].hex()}...")
print(f"  Ciphertext tail:    ...{recv_ciphertext[-16:].hex()}")
print(f"  Tag (hex):          {recv_tag.hex()}")
print(f"  Tag length:         {len(recv_tag)} bytes")
print("="*70)

  FEC decode: 0 uncorrectable codeword(s) -> bit-exact ciphertext recovered
  PARSED CIPHERTEXT BLOB
  File:               extracted_ciphertext.bin
  Total size:         74449 bytes
  Nonce (hex):        848d9ed271b6a721515588e1
  Nonce length:       12 bytes
  Ciphertext length:  74421 bytes
  Ciphertext head:    f81025f638f2a6379248835bc1018e53...
  Ciphertext tail:    ...775b675a234927bf9a119322cf961a45
  Tag (hex):          79e4ae1c389b984c7f19e8cabb9ed83d
  Tag length:         16 bytes


## Section 5: ChaCha20-Poly1305 Decryption Function

This is the exact inverse of Module 2's `encrypt_chacha20_poly1305()`.
The internal algorithm:

1. **Initialize ChaCha20** with the 256-bit key and 96-bit nonce.
2. **Generate one keystream block** to derive the Poly1305 one-time key.
3. **Authenticate** by recomputing Poly1305 over `(AAD ‖ pad ‖ Ct ‖ pad ‖ len(AAD) ‖ len(Ct))`.
4. **Constant-time compare** the recomputed tag against the received tag.
5. If the comparison fails → **raise `ValueError`** (no plaintext returned).
6. If it succeeds → run ChaCha20 in counter mode to produce plaintext.

Steps 4 and 5 are the AEAD security guarantee: PyCryptodome will *never*
return partial plaintext on a failed tag check, which prevents the
classic "decrypt first, verify later" timing-attack class.

In [6]:
def decrypt_chacha20_poly1305(key, nonce, ciphertext, tag, aad=b"QVSC-Module2"):
    """
    Decrypt and authenticate a ChaCha20-Poly1305 ciphertext.
    
    Mirrors Module 2's encrypt_chacha20_poly1305() exactly.
    
    Parameters:
        key        : bytes (32) — symmetric key from Module 1
        nonce      : bytes (12) — nonce extracted from the blob
        ciphertext : bytes      — encrypted payload
        tag        : bytes (16) — Poly1305 authentication tag
        aad        : bytes      — must match the AAD from Module 2
    
    Returns:
        plaintext  : bytes — decrypted message
    
    Raises:
        ValueError — if Poly1305 tag verification fails (data corrupted)
    """
    # Step 1: Initialize cipher with the same key + nonce used for encryption
    cipher = ChaCha20_Poly1305.new(key=key, nonce=nonce)
    
    # Step 2: Feed the AAD to the MAC computation (must match encrypter's AAD)
    cipher.update(aad)
    
    # Step 3: decrypt_and_verify is atomic — either both succeed or it raises
    try:
        plaintext = cipher.decrypt_and_verify(ciphertext, tag)
        return plaintext
    except ValueError as e:
        raise ValueError(
            "Poly1305 tag verification FAILED. The recovered ciphertext "
            "is not byte-identical to the original — decryption rejected."
        ) from e


print("Decryption function defined.")

Decryption function defined.


## Section 6: Run Decryption and Verify Integrity

This is the moment of truth for the entire QVSC pipeline. The extracted
payload has already been Reed-Solomon FEC-decoded (in the cell above),
so the residual bit errors left by H.264 compression are corrected. If
any error remained that FEC could NOT correct, the Poly1305 check will
fail here.

A failure means the receiver knows with cryptographic certainty that
the message has been corrupted (or tampered with), and rejects it.

A success means: QP-controlled compression plus the Reed-Solomon FEC
layer delivered a byte-exact ciphertext, and ChaCha20-Poly1305 verified
it — confirming end-to-end integrity.

In [7]:
# ── Attempt decryption ────────────────────────────────────────────────
decryption_succeeded = False
decrypted_data       = None
failure_reason       = None

try:
    decrypted_data = decrypt_chacha20_poly1305(
        key, recv_nonce, recv_ciphertext, recv_tag, AAD
    )
    decryption_succeeded = True
    
    print("="*70)
    print("  ✓ DECRYPTION SUCCEEDED — Poly1305 tag VERIFIED")
    print("="*70)
    print(f"  Plaintext length:  {len(decrypted_data)} bytes")
    print(f"  Plaintext head:    {decrypted_data[:16].hex()}...")
    print()
    print(f"  This proves end-to-end integrity of the QVSC pipeline:")
    print(f"    • Module 1 — quantum key matches sender's key")
    print(f"    • Modules 2/4 — AAD and nonce binding correct")
    print(f"    • Modules 4/5 — payload survived QP H.264 compression")
    print(f"    • FEC          — Reed-Solomon corrected residual errors to byte-exact")
    print("="*70)

except ValueError as e:
    failure_reason = str(e)
    
    print("="*70)
    print("  ✗ DECRYPTION FAILED — Poly1305 tag MISMATCH")
    print("="*70)
    print(f"  Reason: {failure_reason}")
    print()
    print(f"  Possible causes:")
    print(f"    1. Bit errors exceeded the FEC correction budget (most common)")
    print(f"       → Lower QP in Module 4 (less aggressive compression)")
    print(f"       → Increase RS_PARITY in fec_module.py for more correction")
    print(f"    2. Mask mismatch — Module 5 must extract with the sender's")
    print(f"       roi_masks.npy (the system is non-blind); ensure that file")
    print(f"       matches the embedded video")
    print(f"    3. Wrong key — quantum_key.bin differs from sender's")
    print(f"    4. AAD mismatch — must be exactly b'QVSC-Module2'")
    print(f"    5. Nonce corruption — first 12 bytes of blob were altered")
    print("="*70)

  ✓ DECRYPTION SUCCEEDED — Poly1305 tag VERIFIED
  Plaintext length:  74421 bytes
  Plaintext head:    00000002ffd8ffe000104a4649460001...

  This proves end-to-end integrity of the QVSC pipeline:
    • Module 1 — quantum key matches sender's key
    • Modules 2/4 — AAD and nonce binding correct
    • Modules 4/5 — payload survived QP H.264 compression
    • FEC          — Reed-Solomon corrected residual errors to byte-exact


## Section 7: Parse the Type Header and Save the Recovered Message

Module 2 prepended a 4-byte big-endian header to the plaintext before
encryption:

| Header value     | Meaning |
|------------------|---------|
| `0x00000001`     | UTF-8 text |
| `0x00000002`     | Binary image data |

We strip the header here and route the body to the appropriate output
handler. Because the header is *inside* the encrypted region, an
attacker cannot manipulate it without breaking the Poly1305 tag.

In [8]:
# ── Only proceed if decryption succeeded ──────────────────────────────
recovered_text  = None
msg_type        = None
msg_body        = None
output_path     = None

if decryption_succeeded:
    # Parse 4-byte big-endian type header
    if len(decrypted_data) < 4:
        print(f"  ⚠ Decrypted data too short to contain a type header")
    else:
        msg_type = struct.unpack('>I', decrypted_data[:4])[0]
        msg_body = decrypted_data[4:]
        
        print("="*70)
        print("  RECOVERED PAYLOAD")
        print("="*70)
        print(f"  Type header (raw):  0x{msg_type:08X}")
        print(f"  Body length:        {len(msg_body)} bytes")
        print()
        
        # ── TEXT (Type 1) ─────────────────────────────────────────────
        if msg_type == TYPE_TEXT:
            print(f"  Type:               TEXT (UTF-8)")
            try:
                recovered_text = msg_body.decode('utf-8')
                
                with open(OUTPUT_TEXT_FILE, 'w', encoding='utf-8') as f:
                    f.write(recovered_text)
                output_path = OUTPUT_TEXT_FILE
                
                print(f"  Saved to:           {OUTPUT_TEXT_FILE}")
                print()
                print(f"  ──────── Recovered Message ────────")
                # Truncate display if very long
                display_text = recovered_text if len(recovered_text) <= 500 \
                               else recovered_text[:500] + ' ... [truncated]'
                print(f"  {display_text}")
                print(f"  ───────────────────────────────────")
            
            except UnicodeDecodeError:
                print(f"  ⚠ Body is not valid UTF-8 — saving as raw bytes")
                with open(OUTPUT_TEXT_FILE + '.bin', 'wb') as f:
                    f.write(msg_body)
                output_path = OUTPUT_TEXT_FILE + '.bin'
        
        # ── IMAGE (Type 2) ────────────────────────────────────────────
        elif msg_type == TYPE_IMAGE:
            print(f"  Type:               IMAGE (binary)")
            
            # Try to detect format from magic bytes
            magic = msg_body[:4] if len(msg_body) >= 4 else b''
            if   magic[:3] == b'\xff\xd8\xff': ext = '.jpg'
            elif magic == b'\x89PNG':         ext = '.png'
            elif magic[:3] == b'GIF':         ext = '.gif'
            elif magic[:2] == b'BM':          ext = '.bmp'
            else:                              ext = '.bin'
            
            output_path = OUTPUT_IMAGE_FILE.replace('.bin', ext)
            with open(output_path, 'wb') as f:
                f.write(msg_body)
            
            print(f"  Detected format:    {ext}")
            print(f"  Saved to:           {output_path}")
        
        # ── UNKNOWN TYPE ──────────────────────────────────────────────
        else:
            print(f"  ⚠ UNKNOWN type header (0x{msg_type:08X})")
            print(f"    Saving raw decrypted body for inspection.")
            output_path = 'recovered_unknown.bin'
            with open(output_path, 'wb') as f:
                f.write(msg_body)
        
        print("="*70)
else:
    print("  Skipping payload parsing — decryption failed.")

  RECOVERED PAYLOAD
  Type header (raw):  0x00000002
  Body length:        74417 bytes

  Type:               IMAGE (binary)
  Detected format:    .jpg
  Saved to:           recovered_image.jpg


## Section 8: End-to-End Round-Trip Validation (Optional)

If you have access to the original plaintext (Module 2's input), this
section verifies the full QVSC round-trip is byte-perfect:

```
Sender:  message → M2 encrypt → M4 embed → H.264 compress
Channel: <stego video transmitted>
Receiver: H.264 decode → M5 extract → M6 decrypt → message ?
```

This check confirms that the entire 6-module pipeline preserves the
original message bit-for-bit — the headline result for the thesis.

In [9]:
# ── Optional: compare against original plaintext ─────────────────────
roundtrip_match = None

if decryption_succeeded and ORIGINAL_PLAINTEXT_FILE \
   and os.path.exists(ORIGINAL_PLAINTEXT_FILE):
    
    with open(ORIGINAL_PLAINTEXT_FILE, 'rb') as f:
        original_bytes = f.read()
    
    # The original file might not have the type header — handle both cases
    if msg_body == original_bytes:
        roundtrip_match = True
        compared_against = "raw body"
    elif decrypted_data == original_bytes:
        roundtrip_match = True
        compared_against = "header + body"
    else:
        roundtrip_match = False
        compared_against = "neither variant"
    
    print("="*70)
    print("  END-TO-END ROUND-TRIP VALIDATION")
    print("="*70)
    print(f"  Original file:       {ORIGINAL_PLAINTEXT_FILE}")
    print(f"  Original size:       {len(original_bytes)} bytes")
    print(f"  Recovered body:      {len(msg_body) if msg_body else 0} bytes")
    print(f"  Comparison:          {compared_against}")
    print()
    if roundtrip_match:
        print(f"  ✓ BYTE-PERFECT ROUND-TRIP — QVSC pipeline preserves message")
        print(f"    integrity end-to-end through H.264 compression.")
    else:
        print(f"  ✗ Round-trip mismatch")
        print(f"    Note: this should NEVER happen if Poly1305 verification")
        print(f"    succeeded — AEAD guarantees byte-identity.")
    print("="*70)

elif decryption_succeeded:
    print("  (No original plaintext provided — skipping round-trip check.)")
    print("  The successful Poly1305 verification above is itself proof of")
    print("  byte-perfect recovery. AEAD's integrity guarantee is binary:")
    print("  if the tag matches, every single bit was preserved.")

  (No original plaintext provided — skipping round-trip check.)
  The successful Poly1305 verification above is itself proof of
  byte-perfect recovery. AEAD's integrity guarantee is binary:
  if the tag matches, every single bit was preserved.


## Section 9: Save Decryption Report for Module 7

Module 7 (Quality Benchmarking) will pull this report alongside the
embedding metadata and extraction report to produce the final results
tables for your thesis.

In [10]:
decryption_report = {
    # Inputs
    'extracted_ciphertext':  EXTRACTED_CIPHERTEXT,
    'key_file':              KEY_FILE,
    'key_fingerprint':       key_fingerprint,
    'aad':                   AAD,
    
    # Parsed blob structure
    'nonce':                 recv_nonce.hex(),
    'ciphertext_size':       len(recv_ciphertext),
    'tag':                   recv_tag.hex(),
    
    # Decryption outcome
    'decryption_succeeded':  decryption_succeeded,
    'failure_reason':        failure_reason,
    'plaintext_size':        len(decrypted_data) if decrypted_data else 0,
    
    # Payload information
    'msg_type':              msg_type,
    'msg_type_name':         (
        'TEXT'    if msg_type == TYPE_TEXT  else
        'IMAGE'   if msg_type == TYPE_IMAGE else
        'UNKNOWN' if msg_type is not None   else
        None
    ),
    'msg_body_size':         len(msg_body) if msg_body else 0,
    'output_file':           output_path,
    
    # Round-trip diagnostic (None if not validated)
    'roundtrip_match':       roundtrip_match,
}

np.save(DECRYPTION_REPORT, decryption_report)
print(f"Decryption report saved to '{DECRYPTION_REPORT}'")
print()
for k, v in decryption_report.items():
    if isinstance(v, str) and len(v) > 60:
        print(f"  {k:22s}: {v[:60]}...")
    else:
        print(f"  {k:22s}: {v}")

Decryption report saved to 'decryption_report.npy'

  extracted_ciphertext  : extracted_ciphertext.bin
  key_file              : quantum_key.bin
  key_fingerprint       : 0deecb03eeeb9ce0
  aad                   : b'QVSC-Module2'
  nonce                 : 848d9ed271b6a721515588e1
  ciphertext_size       : 74421
  tag                   : 79e4ae1c389b984c7f19e8cabb9ed83d
  decryption_succeeded  : True
  failure_reason        : None
  plaintext_size        : 74421
  msg_type              : 2
  msg_type_name         : IMAGE
  msg_body_size         : 74417
  output_file           : recovered_image.jpg
  roundtrip_match       : None


## Section 10: Summary

### Module 6 — ChaCha20-Poly1305 Decryption — Complete

This module closes the cryptographic loop of the QVSC pipeline. Every
guarantee made by Module 2 (confidentiality, integrity, authenticity)
is verified here. If the Poly1305 tag check passes, the receiver knows
with cryptographic certainty that the recovered message is identical
to what the sender encrypted — no bit flips, no tampering, no replay.

| Property | Mechanism |
|----------|-----------|
| Confidentiality | ChaCha20 stream cipher, 256-bit key |
| Integrity       | Poly1305 MAC over (AAD ‖ Ciphertext) |
| Authenticity    | Tag verification using shared QKD-derived key |
| Replay resistance | AAD = `b"QVSC-Module2"` binds to this protocol |
| Key strength    | 256 bits derived from E91 entanglement + SHA3 |

### Pipeline status

```
[M1] E91 QKD → 256-bit key                       ✓ DONE
[M2] ChaCha20-Poly1305 encryption                 ✓ DONE
[FEC] Reed-Solomon + interleaving wrap            ✓ DONE
[M3] Canny ROI detection                          ✓ DONE
[M4] DCT-QIM embedding (Y, informed) + QP H.264   ✓ DONE
[M5] DCT-QIM extraction (sender masks, non-blind) ✓ DONE
[M6] FEC decode → ChaCha20-Poly1305 verify        ✓ DONE  ← (this notebook)
[M7] PSNR / SSIM / BER benchmarking + figures     ⏳ NEXT
```

### What feeds into Module 7

Module 7 will collect three reports to produce the thesis results section:

| Source | File | Provides |
|--------|------|----------|
| Module 4 | `embedding_metadata.npy`  | PSNR, SSIM, sender-side BER, embedding params |
| Module 5 | `extraction_report.npy`   | Post-compression BER, mask agreement, per-frame stats |
| Module 6 | `decryption_report.npy`   | End-to-end success/failure, round-trip validation |

Module 7 will also re-run the full pipeline at multiple QP values
(18, 23, 28, ...) to populate the headline table. BER below is the
RAW channel BER; the FEC layer corrects it so Decryption stays PASS
until the error rate exceeds the Reed-Solomon budget:

```
┌───────┬──────────┬────────┬────────┬────────────┐
│  QP   │ raw BER  │  PSNR  │  SSIM  │ Decryption │
├───────┼──────────┼────────┼────────┼────────────┤
│   18  │  ~0.0%   │ ≥48 dB │ ≥0.99  │   PASS     │
│   23  │  ~0.2%   │ ≥48 dB │ ≥0.99  │   PASS     │
│   28  │  ?.??%   │ ≥40 dB │ ≥0.99  │   PASS?    │
│   ..  │  rising  │   ..   │   ..   │ PASS→FAIL  │
└───────┴──────────┴────────┴────────┴────────────┘
```

### Key takeaway for the thesis

Module 6 is what gives the QVSC framework its **cryptographic teeth**.
Without it, an attacker who substitutes their own stego video could
deliver arbitrary content. With Poly1305 verification gated on a
QKD-derived key, the receiver has end-to-end provable security —
quantum-detected eavesdropping (Module 1's CHSH test) plus
classical-but-strong AEAD integrity.